Verifing installations

In [1]:
import sys
import numpy as np
import sounddevice as sd
import soundfile as sf
import requests

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("SoundDevice:", sd.__version__)
print("SoundFile:", sf.__version__)

print("\nAll imports successful!")

Python: d:\Vscode\VoiceAiLedger\venv\Scripts\python.exe
NumPy: 2.2.6
SoundDevice: 0.5.6
SoundFile: 0.14.0

All imports successful!


Test


In [ ]:
import sounddevice as sd

print(sd.query_devices())

   0 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  1 KA270 G0 (NVIDIA High Definitio, MME (0 in, 2 out)
   2 Digital Audio (S/PDIF) (High De, MME (0 in, 2 out)
   3 Speakers (High Definition Audio, MME (0 in, 2 out)
   4 Primary Sound Driver, Windows DirectSound (0 in, 2 out)
   5 KA270 G0 (NVIDIA High Definition Audio), Windows DirectSound (0 in, 2 out)
   6 Digital Audio (S/PDIF) (High Definition Audio Device), Windows DirectSound (0 in, 2 out)
   7 Speakers (High Definition Audio Device), Windows DirectSound (0 in, 2 out)
   8 Digital Audio (S/PDIF) (High Definition Audio Device), Windows WASAPI (0 in, 2 out)
   9 Speakers (High Definition Audio Device), Windows WASAPI (0 in, 2 out)
  10 KA270 G0 (NVIDIA High Definition Audio), Windows WASAPI (0 in, 2 out)
  11 Speakers (HD Audio Speaker), Windows WDM-KS (0 in, 2 out)
  12 SPDIF Out (HD Audio SPDIF out), Windows WDM-KS (0 in, 2 out)
  13 Output (NVIDIA High Definition Audio), Windows WDM-KS (0 in, 2 out)


In [ ]:
import sounddevice as sd
import soundfile as sf
import numpy as np
from pathlib import Path

# Recording settings
DEVICE_ID = 1
SAMPLE_RATE = 16000
DURATION = 5

print("Speak something after recording starts...")

audio = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32",
    device=DEVICE_ID
    
)

sd.wait()

# Save recording
voice_folder = Path.cwd()
recordings_folder = voice_folder / "Recordings"
recordings_folder.mkdir(exist_ok=True)

file_path = recordings_folder / "test_audio.wav"
sf.write(file_path, audio, SAMPLE_RATE)

print("Recording saved:", file_path)
print("Audio shape:", audio.shape)

Speak something after recording starts...
Recording saved: d:\Vscode\VoiceAiLedger\Voice\Recordings\test_audio.wav
Audio shape: (80000, 1)


STT CHecking

In [ ]:
from App.Services.stt import transcribe_audio
import sounddevice as sd
import soundfile as sf
import numpy as np
import io

buffer = io.BytesIO()

# Recording settings
DEVICE_ID = 1
SAMPLE_RATE = 16000
DURATION = 5

print("Speak something after recording starts...")

audio = sd.rec(
    int(DURATION * SAMPLE_RATE),
    samplerate=SAMPLE_RATE,
    channels=1,
    dtype="float32",
    device=DEVICE_ID
)

sd.wait()
sf.write(buffer, audio, SAMPLE_RATE,format='wav')

trasnscript=transcribe_audio(buffer.getvalue(),"ta-IN")



Speak something after recording starts...
ஹலோ நான் இங்கே இருக்கேன் நீங்கள் எங்கே இருக்கீங்க


Sarvam Test

In [ ]:
from sarvamai import SarvamAI
import os
from dotenv import load_dotenv
load_dotenv()
client = SarvamAI(
    api_subscription_key=os.getenv("NLU"),
)
response = client.chat.completions(
    model="sarvam-105b",
    messages=[
       {"role": "system", "content": "you are a ledger assistant for a shopkeeper. Return the output in JSON format with customer name, product, amount paid, or amount owed."},
        {"role": "user", "content": "ரவி 500 ரூபாய் கடன் வாங்கினார்"}
    ],
    temperature=0.5,
    top_p=1,
    max_tokens=2000,
)
print(response.choices[0].message.content)



```json
{
  "customer_name": "Ravi",
  "product": null,
  "amount_paid": 0,
  "amount_owed": 500
}
```
id='20261103_261c6f65-727a-4b3b-a54b-347c8d0f6854' choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionResponseMessage(content='\n```json\n{\n  "customer_name": "Ravi",\n  "product": null,\n  "amount_paid": 0,\n  "amount_owed": 500\n}\n```', refusal=None, reasoning_content=None, role='assistant', tool_calls=None))] created=1793724819 model='sarvam-105b' object='chat.completion' service_tier=None system_fingerprint='vllm-0.29.1rc1.dev145+gfe61f3317-tp2-aec15aed' usage=CompletionUsage(completion_tokens=1663, prompt_tokens=54, total_tokens=1717, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=1612, accepted_prediction_tokens=None, audio_tokens=None, rejected_prediction_tokens=None), prompt_tokens_details=CompletionUsagePromptTokensDetails(cached_tokens=0, audio_tokens=None, created_cache_tokens=32, multimodal_tokens=None))


NLU FILE test

In [22]:
from App.Services.nlu import nlu
query="முருகன் தன்னுடைய கடனில் 500 ரூபாய் கொடுத்தார்."
print(nlu(query))

{'customer_name': 'Murugan', 'transaction_type': 'repayment', 'items': [], 'amount_paid': 500, 'payment_method': None}


In [30]:
import importlib
import App.Services.nlu as nlu_module

importlib.reload(nlu_module)

nlu_correction = nlu_module.nlu_correction

In [31]:
from App.Services.nlu import nlu_correction

current_transaction = {
    "customer_name": "Ravi",
    "transaction_type": "purchase",
    "items": [
        {
            "product_name": "rice",
            "quantity": 2,
            "unit": "kg"
        },
        {
            "product_name": "oil",
            "quantity": 1,
            "unit": "litre"
        }
    ],
    "amount_paid": 100,
    "payment_method": None
}

result = nlu_correction(
    "Actually rice should be 3 kg, not 2 kg",
    current_transaction
)

print(result)


{'status': 'updated', 'transaction': {'customer_name': 'Ravi', 'transaction_type': 'purchase', 'items': [{'product_name': 'rice', 'quantity': 3, 'unit': 'kg'}, {'product_name': 'oil', 'quantity': 1, 'unit': 'litre'}], 'amount_paid': 100, 'payment_method': None}}


In [32]:
result = nlu_correction(
    "Yes, everything is correct",
    current_transaction
)

print(result)

{'status': 'confirmed', 'transaction': {'customer_name': 'Ravi', 'transaction_type': 'purchase', 'items': [{'product_name': 'rice', 'quantity': 2, 'unit': 'kg'}, {'product_name': 'oil', 'quantity': 1, 'unit': 'litre'}], 'amount_paid': 100, 'payment_method': None}}


In [33]:
result = nlu_correction(
    "Cancel this transaction",
    current_transaction
)

print(result)

{'status': 'cancelled', 'transaction': {'customer_name': 'Ravi', 'transaction_type': 'purchase', 'items': [{'product_name': 'rice', 'quantity': 2, 'unit': 'kg'}, {'product_name': 'oil', 'quantity': 1, 'unit': 'litre'}], 'amount_paid': 100, 'payment_method': None}}


In [34]:
result = nlu_correction(
    "Hmm, maybe that one",
    current_transaction
)

print(result)

{'status': 'unclear', 'transaction': {'customer_name': 'Ravi', 'transaction_type': 'purchase', 'items': [{'product_name': 'rice', 'quantity': 2, 'unit': 'kg'}, {'product_name': 'oil', 'quantity': 1, 'unit': 'litre'}], 'amount_paid': 100, 'payment_method': None}}


TTS testing

In [24]:
from App.Services.tts import tts
from IPython.display import Audio, display
text="நல்ல நேரம், நீங்கள் எப்படி இருக்கிறீர்கள்?"
language="ta-IN"

audio_buffer = tts(text, language)

display(Audio(data=audio_buffer.getvalue(), rate=48000))



validation test

In [35]:
from App.Services.validation import validate_transaction

corrected_transaction = result["transaction"]

validation_result = validate_transaction(corrected_transaction)

print(validation_result)

{'valid': True, 'missing_fields': []}


pipline testing


In [1]:
from App.Services.ai_pipeline import process_turn

print("AI pipeline imported successfully")

AI pipeline imported successfully
AI pipeline imported successfully


In [3]:
from App.Services.ai_pipeline import process_turn

with open("Recordings/test_audio.wav", "rb") as f:
    audio_bytes = f.read()

result = process_turn(
    conversation_id="test_001",
    audio=audio_bytes,
    language_code="ta-IN"
)

print("Pipeline result:", result)

Pipeline result: {'status': 'unclear', 'message': "What is the customer's name?", 'audio': b'RIFF\xff\xff\xff\xffWAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80\xbb\x00\x00\x00w\x01\x00\x02\x00\x10\x00data\xff\xff\xff\xff\xfb\xff\xf7\xff\xf7\xff\xf6\xff\xf5\xff\xf6\xff\xf4\xff\xf5\xff\xf5\xff\xf5\xff\xf3\xff\xf2\xff\xf4\xff\xf1\xff\xf0\xff\xf2\xff\xf3\xff\xf2\xff\xf1\xff\xf3\xff\xf2\xff\xf1\xff\xf2\xff\xf0\xff\xf1\xff\xf1\xff\xf1\xff\xf2\xff\xf3\xff\xf3\xff\xf3\xff\xf4\xff\xf3\xff\xf2\xff\xf3\xff\xf4\xff\xf4\xff\xf5\xff\xf6\xff\xf7\xff\xf8\xff\xf9\xff\xf9\xff\xfa\xff\xfa\xff\xfb\xff\xfb\xff\xfb\xff\xfc\xff\xfc\xff\xfd\xff\xff\xff\xfd\xff\xfd\xff\xfd\xff\xfe\xff\xff\xff\xfe\xff\xfe\xff\xff\xff\xfe\xff\xff\xff\xff\xff\xff\xff\x01\x00\x01\x00\x01\x00\x03\x00\x02\x00\x02\x00\x04\x00\x04\x00\x04\x00\x05\x00\x07\x00\x06\x00\x07\x00\x08\x00\x08\x00\x08\x00\x08\x00\x08\x00\x08\x00\t\x00\x08\x00\x08\x00\x08\x00\x08\x00\x08\x00\x08\x00\x08\x00\x07\x00\x07\x00\x07\x00\x06\x00\x06\x00\x05\x00\x04\x00

In [5]:
from App.Services.ai_pipeline import conversations

print(conversations["test_001"])

{'transaction': {'customer_name': None, 'transaction_type': 'unclear', 'items': [], 'amount_paid': None, 'payment_method': None}, 'phase': 'collecting', 'last_question': "What is the customer's name?", 'language_code': 'ta-IN', 'awaiting_field': 'customer_name'}


In [1]:
from App.Services.ai_pipeline import process_turn

with open("Recordings/test_audio.wav", "rb") as f:
    audio_bytes = f.read()

result = process_turn(
    conversation_id="test_002",
    audio=audio_bytes,
    language_code="ta-IN"
)

print("Status:", result["status"])
print("Message:", result["message"])

Status: needs_information
Message: What is the customer's name?


In [2]:
from App.Services.ai_pipeline import conversations

print(conversations["test_002"])

{'transaction': {'customer_name': None, 'transaction_type': 'unclear', 'items': [], 'amount_paid': None, 'payment_method': None}, 'phase': 'collecting', 'last_question': "What is the customer's name?", 'language_code': 'ta-IN', 'awaiting_field': 'customer_name'}


In [1]:
import importlib
import App.Services.ai_pipeline as pipeline

importlib.reload(pipeline)

import inspect
print(inspect.getsource(pipeline.process_turn))

def process_turn(conversation_id, audio, language_code):

    # 1. Convert audio to text
    transcript = transcribe_audio(audio, language_code)

    if isinstance(transcript, tuple):
        return {
            "status": "error",
            "message": str(transcript),
            "audio": None
        }

    # 2. Retrieve existing conversation
    conversation = get_conversation(conversation_id)

    # 3. New conversation
    if conversation is None:

        start_conversation(conversation_id, language_code)

        transaction = nlu(transcript)

        update_transaction(conversation_id, transaction)

    # 4. Existing conversation: process correction
    else:

        transaction = conversation["transaction"]
        awaiting_field = conversation["awaiting_field"]


        correction = nlu_correction(
            transcript,
            transaction,
            awaiting_field,
            phase=conversation["phase"]
        )

        correction_status = correction.get("statu

In [ ]:
import inspect
print(inspect.getsource(pipeline.process_turn))


In [6]:
import importlib
import App.Services.ai_pipeline as pipeline

importlib.reload(pipeline)

print("Reload successful")
print("Unclear handler:", "I couldn't understand the transaction" in 
      __import__("inspect").getsource(pipeline.process_turn))

Reload successful
Unclear handler: True


In [ ]:
import soundfile as sf
import io

audio_data, sample_rate = sf.read("Recordings/test_audio.wav")

buffer = io.BytesIO()
sf.write(buffer, audio_data, sample_rate, format="WAV")

audio_bytes = buffer.getvalue()

result = pipeline.process_turn(
    "test_003",
    audio_bytes,
    "ta-IN"
)

print("Status:", result["status"])
print("Message:", result.get("message"))
print("Transaction:", result.get("transaction"))
print("Audio generated:", result.get("audio") is not None)

In [8]:
from App.Services.stt import transcribe_audio

transcript = transcribe_audio(audio_bytes, "ta-IN")

print("Transcript:", transcript)

Transcript: ரவி ரெண்டு கிலோ அரிசி வாங்கினார்


All file Re Import

In [1]:
import importlib

from App.Services import stt, nlu, validation, tts, ai_pipeline

# Reload modules in dependency order
importlib.reload(stt)
importlib.reload(nlu)
importlib.reload(validation)
importlib.reload(tts)
importlib.reload(ai_pipeline)

print("All service modules reloaded successfully!")

# Verify the files being imported
print("\nLoaded module paths:")
print("STT:        ", stt.__file__)
print("NLU:        ", nlu.__file__)
print("Validation: ", validation.__file__)
print("TTS:        ", tts.__file__)
print("AI Pipeline:", ai_pipeline.__file__)

All service modules reloaded successfully!

Loaded module paths:
STT:         d:\Vscode\VoiceAiLedger\Voice\App\Services\stt.py
NLU:         d:\Vscode\VoiceAiLedger\Voice\App\Services\nlu.py
Validation:  d:\Vscode\VoiceAiLedger\Voice\App\Services\validation.py
TTS:         d:\Vscode\VoiceAiLedger\Voice\App\Services\tts.py
AI Pipeline: d:\Vscode\VoiceAiLedger\Voice\App\Services\ai_pipeline.py


Full piple test


In [4]:
import sounddevice as sd
import soundfile as sf
import io
import uuid

from App.Services import ai_pipeline as pipeline

DEVICE_ID = 1
SAMPLE_RATE = 16000
DURATION = 5
LANGUAGE_CODE = "ta-IN"

# One conversation ID for the entire conversation
conversation_id = str(uuid.uuid4())

def record_and_process():

    print("\n🎤 Speak now...")

    audio = sd.rec(
        int(DURATION * SAMPLE_RATE),
        samplerate=SAMPLE_RATE,
        channels=2,
        dtype="float32",
        device=DEVICE_ID
    )

    sd.wait()

    # Convert recording to WAV bytes in memory
    buffer = io.BytesIO()
    sf.write(buffer, audio, SAMPLE_RATE, format="WAV")
    audio_bytes = buffer.getvalue()

    print("Processing audio...")

    result = pipeline.process_turn(
        conversation_id,
        audio_bytes,
        LANGUAGE_CODE
    )

    print("\nStatus:", result.get("status"))
    print("Message:", result.get("message"))
    print("Transaction:", pipeline.get_conversation(conversation_id).get("transaction")
          if pipeline.get_conversation(conversation_id) else None)
    print("Audio generated:", result.get("audio") is not None)

    # Play the generated TTS response
    if result.get("audio"):
        response_audio = io.BytesIO(result["audio"])
        spoken_audio, sr = sf.read(response_audio)
        sd.play(spoken_audio, sr)
        sd.wait()

    return result

In [23]:
result = record_and_process()


🎤 Speak now...
Processing audio...
HTTP Status: 200
Content-Type: application/json
Response: '{"success":true,"request_id":"01a1aa57-93ce-7e69-8756-38076ac5833d","transcript":"ஆம் உறுதிப்படுத்தலாம்","model":"prisma-v2.5","processing_time":0.16896462440490723,"end_to_end_latency":0.17303752899169922}'

Status: confirmed
Message: None
Transaction: {'customer_name': 'Suresh', 'transaction_type': 'purchase', 'items': [{'product_name': 'tomato', 'quantity': 0.5, 'unit': 'kg'}], 'amount_paid': 100, 'payment_mentioned': True, 'payment_method': None}
Audio generated: False


In [11]:
conversation = pipeline.get_conversation(conversation_id)

print("Conversation phase:", conversation["phase"])
print("Last question:", conversation["last_question"])
print("Awaiting field:", conversation["awaiting_field"])

Conversation phase: confirmed
Last question: Ramesh 10 கிலோ Tomato வாங்கியுள்ளார். ₹500 செலுத்தியுள்ளார். இந்தப் பரிவர்த்தனையை உறுதிப்படுத்தலாமா?
Awaiting field: None


In [18]:
conversation_id = str(uuid.uuid4())
print("New conversation:", conversation_id)

New conversation: bb0157ac-e16f-4c29-8619-a73a688d739f


In [49]:
conversation = pipeline.get_conversation(conversation_id)

print(conversation["phase"])
print(conversation["awaiting_field"])

collecting
items[0].quantity
